# Bolero Isolation Lab: Depth, Importance, Detection, and Boundary Proposals

**Standalone first-stage research notebook.** This notebook does not import, overwrite, or depend on the other Bolero notebooks. It reads an existing Imagenette validation folder, runs frozen pretrained models, and writes all generated images and reports to `outputs/bolero_isolation/`.

## Research question

Can relative depth, a ResNet-50 class-importance map, DETR detections, and RGB/depth boundary evidence be combined into a useful, irregular subject-region proposal?

The grid has two purposes:

1. Make depth and importance spatial structure easy to inspect.
2. Create a coarse, connected region proposal seeded by class importance and constrained by likely RGB/depth boundaries. Optional GrabCut refinement tests whether pixel-level contours can improve on the grid proposal.

**Important:** pseudo-thermal colors are only a visualization of relative depth, not temperature or calibrated physical distance. Grad-CAM is a model-dependent importance cue, not a true object mask. DETR supplies bounding boxes, not exact outlines. Result overlays are saved for people to inspect and are never passed back into ResNet as altered input images.

## Scope for this version

- Establish subject-isolation diagnostics on clean images.
- Save the original RGB image, depth maps, importance map, detections, grid proposal, and contour overlay separately.
- Compare grid masks and optional boundary refinement using manually annotated masks when available.
- Do **not** reconstruct corrupted images yet. Do not treat any automatically generated mask as ground truth.

Run the notebook from top to bottom using **Run All**. The initial configuration cell clears only this notebook's tagged output folder. It does not clear datasets, model caches, annotations, or other Bolero results.

## Cell 1: Configure the independent experiment

In [ ]:
#Cell 1: Intent
# Configure paths, deterministic sampling, the grid, model IDs, and a safely isolated output tag.

from pathlib import Path
import os
import random
import shutil
import sys
import time
import warnings

import numpy as np

SEED = 42
OUTPUT_TAG = "bolero_isolation"  # Change this tag to keep separate named runs.
GRID_SIZE = 50                   # GRID_SIZE x GRID_SIZE cells. Must be >= 10.
IMAGES_PER_CLASS = 2          # None means use all images found in each class folder.
MAX_IMAGES_TOTAL = None          # None means no additional total-image limit.

# Use the existing dataset location if available, otherwise use the local project data folder.
KNOWN_DATASET_PATH = Path(
    r"C:\Users\srohi\VSCodeProjects\bolero_contextual_robustness\data\imagenette2-160\val"
)

_cwd = Path.cwd().resolve()
if _cwd.name.lower() == "notebooks" and (_cwd.parent / "requirements.txt").exists():
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "requirements.txt").exists():
    PROJECT_ROOT = _cwd
else:
    # Best effort for common VS Code working-directory arrangements.
    PROJECT_ROOT = _cwd

_local_dataset = PROJECT_ROOT / "data" / "imagenette2-160" / "val"
if KNOWN_DATASET_PATH.exists():
    DATASET_ROOT = KNOWN_DATASET_PATH
else:
    DATASET_ROOT = _local_dataset

if GRID_SIZE < 10:
    raise ValueError("GRID_SIZE must be at least 10. Recommended first setting: 20.")
if not OUTPUT_TAG or Path(OUTPUT_TAG).name != OUTPUT_TAG or OUTPUT_TAG in {".", ".."}:
    raise ValueError("OUTPUT_TAG must be a simple folder name, not a path.")

OUTPUT_PARENT = PROJECT_ROOT / "outputs"
if OUTPUT_PARENT.is_symlink():
    raise RuntimeError(f"Refusing to use a symlink as the outputs folder: {OUTPUT_PARENT}")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = OUTPUT_PARENT / OUTPUT_TAG
if OUTPUT_ROOT.is_symlink():
    raise RuntimeError(f"Refusing to clear a symlinked output folder: {OUTPUT_ROOT}")
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
PROJECT_ROOT_RESOLVED = PROJECT_ROOT.resolve()
OUTPUT_PARENT_RESOLVED = OUTPUT_PARENT.resolve()
if OUTPUT_ROOT.parent != OUTPUT_PARENT_RESOLVED or PROJECT_ROOT_RESOLVED not in OUTPUT_ROOT.parents:
    raise RuntimeError(f"Safety check failed. Refusing to clear output path: {OUTPUT_ROOT}")

# Clear only the exact tagged output folder. Preserve the folder itself.
if OUTPUT_ROOT.exists():
    for child in OUTPUT_ROOT.iterdir():
        if child.is_symlink() or child.is_file():
            child.unlink()
        elif child.is_dir():
            shutil.rmtree(child)
else:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_DIRS = {
    "original": OUTPUT_ROOT / "original",
    "depth": OUTPUT_ROOT / "depth",
    "illumination": OUTPUT_ROOT / "illumination",
    "importance": OUTPUT_ROOT / "importance",
    "detections": OUTPUT_ROOT / "detections",
    "masks": OUTPUT_ROOT / "grids_and_masks",
    "overlays": OUTPUT_ROOT / "overlays",
    "reports": OUTPUT_ROOT / "reports",
}
for _path in OUTPUT_DIRS.values():
    _path.mkdir(parents=True, exist_ok=True)

# Human annotations live outside the auto-cleared output folder so they survive reruns.
ANNOTATION_ROOT = PROJECT_ROOT / "annotations" / "ground_truth_masks"
ANNOTATION_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print(f"Python: {sys.version.split()[0]} (target: 3.11.9)")
if sys.version_info[:3] != (3, 11, 9):
    warnings.warn("This project targets Python 3.11.9. Continue only if your installed dependencies work in this interpreter.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Output tag: {OUTPUT_TAG}")
print(f"Output root (only this folder is cleared on Run All): {OUTPUT_ROOT}")
print(f"Grid: {GRID_SIZE} x {GRID_SIZE}")
print(f"Persistent manual-mask folder: {ANNOTATION_ROOT}")

## Cell 2: Imports and experiment settings

In [ ]:
#Cell 2: Intent
# Import the supported stack and declare model/candidate-boundary settings in one place.

import json
import math
import traceback
from collections import deque
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation,
    DetrForObjectDetection,
    DetrImageProcessor,
)

# Model checkpoints. They download once and are then stored in the normal local model cache.
RESNET_WEIGHTS = ResNet50_Weights.DEFAULT
DEPTH_MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
DETR_MODEL_ID = "facebook/detr-resnet-50"

DEVICE_SETTING = "auto"  # "auto", "cpu", or "cuda"
if DEVICE_SETTING == "auto":
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elif DEVICE_SETTING in {"cpu", "cuda"}:
    DEVICE = torch.device(DEVICE_SETTING)
else:
    raise ValueError("DEVICE_SETTING must be 'auto', 'cpu', or 'cuda'.")
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but torch.cuda.is_available() is False. Set DEVICE_SETTING='cpu' or install a matching PyTorch build.")

DETR_SCORE_THRESHOLD = 0.35
GRID_SEED_PERCENTILE = 85
MIN_SEED_CAM = 0.25
MIN_CELL_SUPPORT = 0.30
MAX_BOUNDARY_COST = 0.76
ROI_EXPANSION = 0.08
ENABLE_GRABCUT_REFINEMENT = True
GRABCUT_ITERATIONS = 3

# Imagenette folder synsets -> display labels. These labels are used for sampling/reporting only,
# never for selecting the subject region or the DETR box.
IMAGENETTE_LABELS = {
    "n01440764": "tench",
    "n02102040": "English springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}

# Full-frame transform intentionally preserves the entire photograph. No center crop is applied,
# because spatial heatmaps need to correspond to the whole original image. The resulting ResNet
# scores are exploratory full-frame scores, not the standard center-crop ImageNet benchmark.
FULL_FRAME_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"Device: {DEVICE}")
print(f"ResNet weights: {RESNET_WEIGHTS}")
print(f"Depth model: {DEPTH_MODEL_ID}")
print(f"DETR model: {DETR_MODEL_ID}")
print(f"DETR score threshold: {DETR_SCORE_THRESHOLD}")
print(f"Grid region-growth settings: seed p{GRID_SEED_PERCENTILE}, min support {MIN_CELL_SUPPORT}, max boundary cost {MAX_BOUNDARY_COST}")

## Cell 3: Find and deterministically sample the validation images

In [ ]:
#Cell 3: Intent
# Load a stratified, deterministic image list from Imagenette's class-folder layout.

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        "Imagenette validation folder was not found. Set DATASET_ROOT in Cell 1 to the folder "
        "that contains class subfolders such as n01440764, n02102040, and n03417042. "
        f"Current value: {DATASET_ROOT}"
    )

records = []
class_dirs = [p for p in DATASET_ROOT.iterdir() if p.is_dir()]
if class_dirs:
    for class_dir in sorted(class_dirs, key=lambda p: p.name):
        class_files = sorted(p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)
        if not class_files:
            continue
        label = IMAGENETTE_LABELS.get(class_dir.name, class_dir.name)
        shuffled = class_files.copy()
        random.Random(f"{SEED}:{class_dir.name}").shuffle(shuffled)
        if IMAGES_PER_CLASS is not None:
            shuffled = shuffled[:IMAGES_PER_CLASS]
        for path in shuffled:
            item_id = f"{class_dir.name}_{path.stem}"
            records.append({"item_id": item_id, "path": path, "folder_synset": class_dir.name, "folder_label": label})
else:
    # Flat image folders are allowed for quick smoke tests, but they have no class-based stratification.
    flat_files = sorted(p for p in DATASET_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)
    for path in flat_files:
        records.append({"item_id": path.stem, "path": path, "folder_synset": "unknown", "folder_label": "unknown"})

if MAX_IMAGES_TOTAL is not None:
    records = sorted(records, key=lambda x: x["item_id"])
    random.Random(SEED).shuffle(records)
    records = records[:MAX_IMAGES_TOTAL]

if not records:
    raise RuntimeError(f"No supported images found under {DATASET_ROOT}")

records = sorted(records, key=lambda x: x["item_id"])
print(f"Images queued: {len(records)}")
print(pd.Series([r["folder_label"] for r in records]).value_counts().sort_index().to_string())
print("Folder labels are retained only for stratification/reporting; the region proposal uses model outputs and image signals.")

## Cell 4: Load pretrained models

In [ ]:
#Cell 4: Intent
# Load frozen ResNet-50, Depth Anything V2 Small, and DETR. First run downloads the checkpoints.

print("Loading ResNet-50 ImageNet-1K weights...")
resnet_model = resnet50(weights=RESNET_WEIGHTS).to(DEVICE).eval()
imagenet_categories = RESNET_WEIGHTS.meta["categories"]

print("Loading Depth Anything V2 Small...")
depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_ID)
depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_ID).to(DEVICE).eval()

print("Loading DETR ResNet-50 COCO detector...")
# The no_timm revision avoids requiring the timm implementation path for this checkpoint.
detr_processor = DetrImageProcessor.from_pretrained(DETR_MODEL_ID, revision="no_timm")
detr_model = DetrForObjectDetection.from_pretrained(DETR_MODEL_ID, revision="no_timm").to(DEVICE).eval()
detr_id2label = {int(k): str(v) for k, v in detr_model.config.id2label.items()}

# ResNet class names for reporting only; they are not used to pick the region.
imagenet_category_to_index = {str(name).lower(): i for i, name in enumerate(imagenet_categories)}
for synset, name in IMAGENETTE_LABELS.items():
    idx = imagenet_category_to_index.get(name.lower())
    if idx is None:
        print(f"Note: no exact ImageNet-1K category match for folder label '{name}' ({synset}).")

print("Models loaded.")
print("All models are frozen and used for inference only; no training occurs in this notebook.")

## Cell 5: Image, heatmap, detection, and region-refinement helpers

In [ ]:
#Cell 5: Intent
# Define the per-image inference functions and the grid-based boundary-aware region-growing method.


def normalize_robust(array, low=1.0, high=99.0):
    """Normalize an array to [0, 1] using robust percentiles."""
    arr = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype=np.float32)
    vals = arr[finite]
    lo, hi = np.percentile(vals, [low, high])
    if hi <= lo + 1e-8:
        lo, hi = float(vals.min()), float(vals.max())
    out = np.zeros_like(arr, dtype=np.float32)
    if hi > lo + 1e-8:
        out[finite] = np.clip((arr[finite] - lo) / (hi - lo), 0.0, 1.0)
    return out


def resnet_prediction_and_gradcam(image):
    """Return full-frame ResNet top-5 predictions and Grad-CAM aligned to the original image."""
    input_tensor = FULL_FRAME_TRANSFORM(image).unsqueeze(0).to(DEVICE)
    activations, gradients = {}, {}

    def _forward_hook(_module, _inputs, output):
        activations["features"] = output
        output.register_hook(lambda grad: gradients.__setitem__("features", grad))

    handle = resnet_model.layer4.register_forward_hook(_forward_hook)
    try:
        resnet_model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = resnet_model(input_tensor)
            predicted_idx = int(logits[0].argmax().item())
            logits[0, predicted_idx].backward()
        probs = logits.detach().softmax(dim=1)[0]
        top_values, top_indices = torch.topk(probs, k=5)
        top5 = [
            {"class_id": int(class_idx.item()), "label": imagenet_categories[int(class_idx.item())], "score": float(score.item())}
            for score, class_idx in zip(top_values, top_indices)
        ]
        feature_map = activations["features"].detach()
        gradient_map = gradients["features"].detach()
        weights = gradient_map.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * feature_map).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(image.height, image.width), mode="bilinear", align_corners=False)[0, 0]
        cam_np = cam.detach().cpu().numpy()
    finally:
        handle.remove()
        resnet_model.zero_grad(set_to_none=True)

    cam_np = normalize_robust(cam_np, 1, 99)
    return top5, cam_np


def predict_depth(image):
    """Predict relative depth and resize it to the exact original image dimensions."""
    inputs = depth_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        outputs = depth_model(**inputs)
    processed = depth_processor.post_process_depth_estimation(
        outputs,
        target_sizes=[(image.height, image.width)],
    )
    depth = processed[0]["predicted_depth"].detach().float().cpu().numpy()
    depth = np.squeeze(depth)
    if depth.shape != (image.height, image.width):
        depth = cv2.resize(depth.astype(np.float32), (image.width, image.height), interpolation=cv2.INTER_CUBIC)
    return normalize_robust(depth, 1, 99), depth.astype(np.float32)


def predict_detections(image):
    """Return DETR boxes in original-image pixels."""
    inputs = detr_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        outputs = detr_model(**inputs)
    target_sizes = torch.tensor([[image.height, image.width]], device=DEVICE)
    result = detr_processor.post_process_object_detection(
        outputs,
        target_sizes=target_sizes,
        threshold=DETR_SCORE_THRESHOLD,
    )[0]
    dets = []
    for score, label_id, box in zip(result["scores"], result["labels"], result["boxes"]):
        x0, y0, x1, y1 = [float(v) for v in box.detach().cpu().tolist()]
        x0 = int(np.clip(round(x0), 0, image.width - 1))
        x1 = int(np.clip(round(x1), 1, image.width))
        y0 = int(np.clip(round(y0), 0, image.height - 1))
        y1 = int(np.clip(round(y1), 1, image.height))
        if x1 <= x0 or y1 <= y0:
            continue
        label_int = int(label_id.detach().cpu().item())
        dets.append({
            "label_id": label_int,
            "label": detr_id2label.get(label_int, str(label_int)),
            "score": float(score.detach().cpu().item()),
            "box": (x0, y0, x1, y1),
        })
    dets.sort(key=lambda d: d["score"], reverse=True)
    return dets


def likely_coco_labels(resnet_label):
    """Map recognizable ResNet label words to compatible broad COCO categories when possible."""
    text = str(resnet_label).lower()
    result = set()
    dog_terms = ["dog", "spaniel", "terrier", "retriever", "shepherd", "poodle", "pug", "beagle", "hound", "chihuahua", "schnauzer", "husky", "malamute", "clumber", "springer", "basset", "bluetick", "borzoi", "boxer", "briard", "collie", "dingo", "dhole", "papillon", "pekinese", "pomeranian", "rottweiler", "saluki", "samoyed", "schipperke", "setter", "shih-tzu", "whippet", "wolfhound"]
    if any(term in text for term in dog_terms):
        result.add("dog")
    if any(term in text for term in ["truck", "fire engine", "moving van", "tow truck"]):
        result.add("truck")
    if any(term in text for term in ["ball", "golf ball", "soccer ball", "tennis ball", "baseball", "croquet ball", "ping-pong"]):
        result.add("sports ball")
    if "cat" in text or any(term in text for term in ["tabby", "tiger cat", "persian cat", "siamese cat", "egyptian cat", "lynx"]):
        result.add("cat")
    if any(term in text for term in ["church", "bell cote", "monastery", "mosque", "palace", "castle", "stupa"]):
        result.add("building")  # Diagnostic hint only; COCO DETR has no general building category.
    if "bird" in text or any(term in text for term in ["finch", "jay", "eagle", "parrot", "macaw", "cockatoo", "ostrich", "penguin", "hen", "rooster"]):
        result.add("bird")
    return result


def box_cam_mass(cam, box):
    x0, y0, x1, y1 = box
    total = float(cam.sum()) + 1e-8
    return float(cam[y0:y1, x0:x1].sum() / total)


def expand_box(box, width, height, fraction=ROI_EXPANSION):
    x0, y0, x1, y1 = box
    dx = int(round((x1 - x0) * fraction))
    dy = int(round((y1 - y0) * fraction))
    return (
        max(0, x0 - dx), max(0, y0 - dy),
        min(width, x1 + dx), min(height, y1 + dy),
    )


def select_subject_detection(detections, cam, resnet_label, width, height):
    """Select a detection only when class compatibility or CAM-overlap provides some support."""
    compatible_labels = likely_coco_labels(resnet_label)
    enriched = []
    for det in detections:
        box = det["box"]
        area = ((box[2] - box[0]) * (box[3] - box[1])) / float(width * height)
        mass = box_cam_mass(cam, box)
        compatible = det["label"].lower() in compatible_labels
        # Reward importance mass and detector confidence; penalize boxes spanning nearly all pixels.
        area_penalty = max(0.0, (area - 0.75) / 0.25) * 0.15
        rank_score = 0.50 * mass + 0.35 * det["score"] + 0.15 * float(compatible) - area_penalty
        enriched.append({**det, "cam_mass_fraction": mass, "area_fraction": area, "compatible": compatible, "selection_score": rank_score})

    compatible = [d for d in enriched if d["compatible"]]
    if compatible:
        picked = max(compatible, key=lambda d: d["selection_score"])
        return picked, "class-compatible DETR proposal"

    # DETR's COCO ontology does not cover every Imagenette class. An unmatched box is retained only
    # as a hypothesis when the confidence and class-importance overlap are both substantial.
    plausible = [d for d in enriched if d["score"] >= 0.60 and d["cam_mass_fraction"] >= 0.20 and d["area_fraction"] <= 0.85]
    if plausible:
        picked = max(plausible, key=lambda d: d["selection_score"])
        return picked, "CAM-supported unmatched DETR hypothesis"
    return None, "importance-only fallback (no supported DETR box)"


def fallback_cam_box(cam):
    """Create a loose ROI around high-importance pixels when DETR does not give a supported box."""
    threshold = max(0.25, float(np.percentile(cam, 75)))
    ys, xs = np.where(cam >= threshold)
    if len(xs) < 3:
        y, x = np.unravel_index(int(np.argmax(cam)), cam.shape)
        return (max(0, x - 2), max(0, y - 2), min(cam.shape[1], x + 3), min(cam.shape[0], y + 3))
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    return expand_box((x0, y0, x1, y1), cam.shape[1], cam.shape[0], fraction=0.12)


def edge_maps(rgb, depth_norm):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    rgb_edge = normalize_robust(cv2.magnitude(gx, gy), 1, 99)
    dx = cv2.Sobel(depth_norm.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    dy = cv2.Sobel(depth_norm.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    depth_edge = normalize_robust(cv2.magnitude(dx, dy), 1, 99)
    return rgb_edge, depth_edge


def make_grid_proposal(rgb, cam, depth_norm, roi_box, grid_size=GRID_SIZE):
    """Run a boundary-aware, CAM-seeded flood fill on a configurable rectangular grid."""
    height, width = rgb.shape[:2]
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    rgb_edge, depth_edge = edge_maps(rgb, depth_norm)
    y_edges = np.linspace(0, height, grid_size + 1, dtype=int)
    x_edges = np.linspace(0, width, grid_size + 1, dtype=int)

    cam_cells = np.zeros((grid_size, grid_size), dtype=np.float32)
    depth_cells = np.zeros_like(cam_cells)
    edge_cells = np.zeros_like(cam_cells)
    lab_cells = np.zeros((grid_size, grid_size, 3), dtype=np.float32)
    allowed = np.zeros((grid_size, grid_size), dtype=bool)

    x0, y0, x1, y1 = roi_box
    for r in range(grid_size):
        ya, yb = y_edges[r], y_edges[r + 1]
        for c in range(grid_size):
            xa, xb = x_edges[c], x_edges[c + 1]
            if yb <= ya or xb <= xa:
                continue
            cam_cells[r, c] = float(np.mean(cam[ya:yb, xa:xb]))
            depth_cells[r, c] = float(np.mean(depth_norm[ya:yb, xa:xb]))
            edge_cells[r, c] = float(np.mean(rgb_edge[ya:yb, xa:xb]))
            lab_cells[r, c] = np.mean(lab[ya:yb, xa:xb], axis=(0, 1))
            overlap_w = max(0, min(xb, x1) - max(xa, x0))
            overlap_h = max(0, min(yb, y1) - max(ya, y0))
            allowed[r, c] = (overlap_w * overlap_h) > 0

    if not allowed.any():
        allowed[:] = True

    allowed_cam = cam_cells[allowed]
    seed_threshold = max(MIN_SEED_CAM, float(np.percentile(allowed_cam, GRID_SEED_PERCENTILE)))
    seeds = allowed & (cam_cells >= seed_threshold)
    if not seeds.any():
        allowed_positions = np.argwhere(allowed)
        best_flat = int(np.argmax(np.where(allowed, cam_cells, -1.0)))
        seeds.flat[best_flat] = True

    seed_labs = lab_cells[seeds]
    seed_depths = depth_cells[seeds]
    seed_lab = np.median(seed_labs, axis=0) if len(seed_labs) else np.median(lab_cells[allowed], axis=0)
    seed_depth = float(np.median(seed_depths)) if len(seed_depths) else float(np.median(depth_cells[allowed]))
    lab_dist = np.linalg.norm(lab_cells - seed_lab[None, None, :], axis=2)
    color_similarity = np.exp(-lab_dist / 38.0)
    depth_similarity = np.exp(-np.abs(depth_cells - seed_depth) / 0.28)
    support = 0.60 * cam_cells + 0.25 * color_similarity + 0.15 * depth_similarity

    def boundary_cost(r1, c1, r2, c2):
        color_cost = min(float(np.linalg.norm(lab_cells[r1, c1] - lab_cells[r2, c2])) / 80.0, 1.0)
        depth_cost = min(abs(float(depth_cells[r1, c1] - depth_cells[r2, c2])) / 0.40, 1.0)
        if r1 == r2:
            xline = int(x_edges[max(c1, c2)])
            ya, yb = int(y_edges[r1]), int(y_edges[r1 + 1])
            xa, xb = max(0, xline - 2), min(width, xline + 2)
            rgb_barrier = float(np.mean(rgb_edge[ya:yb, xa:xb])) if yb > ya and xb > xa else 0.0
            depth_barrier = float(np.mean(depth_edge[ya:yb, xa:xb])) if yb > ya and xb > xa else 0.0
        else:
            yline = int(y_edges[max(r1, r2)])
            xa, xb = int(x_edges[c1]), int(x_edges[c1 + 1])
            ya, yb = max(0, yline - 2), min(height, yline + 2)
            rgb_barrier = float(np.mean(rgb_edge[ya:yb, xa:xb])) if yb > ya and xb > xa else 0.0
            depth_barrier = float(np.mean(depth_edge[ya:yb, xa:xb])) if yb > ya and xb > xa else 0.0
        return 0.35 * color_cost + 0.25 * depth_cost + 0.25 * rgb_barrier + 0.15 * depth_barrier

    grown = np.zeros((grid_size, grid_size), dtype=bool)
    queue = deque()
    for r, c in np.argwhere(seeds):
        grown[r, c] = True
        queue.append((int(r), int(c)))
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in neighbors:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < grid_size and 0 <= nc < grid_size):
                continue
            if grown[nr, nc] or not allowed[nr, nc]:
                continue
            if support[nr, nc] < MIN_CELL_SUPPORT:
                continue
            if boundary_cost(r, c, nr, nc) > MAX_BOUNDARY_COST:
                continue
            grown[nr, nc] = True
            queue.append((nr, nc))

    # Keep the connected component with the largest combined support and importance mass.
    component_count, component_labels = cv2.connectedComponents(grown.astype(np.uint8), connectivity=4)
    if component_count > 1:
        component_scores = []
        for component_id in range(1, component_count):
            comp = component_labels == component_id
            component_scores.append((float((support[comp] + 0.25 * cam_cells[comp]).sum()), component_id))
        if component_scores:
            best_component = max(component_scores)[1]
            grown = component_labels == best_component

    pixel_mask = np.zeros((height, width), dtype=np.uint8)
    for r in range(grid_size):
        ya, yb = y_edges[r], y_edges[r + 1]
        for c in range(grid_size):
            if grown[r, c]:
                xa, xb = x_edges[c], x_edges[c + 1]
                pixel_mask[ya:yb, xa:xb] = 1

    # Keep a native-resolution mask and a grid-resolution map so the coarse decision is inspectable.
    grid_mask = pixel_mask.astype(bool)
    return {
        "mask": grid_mask,
        "cell_mask": grown,
        "cell_importance": cam_cells,
        "cell_support": support,
        "allowed_cells": allowed,
        "roi_box": roi_box,
        "rgb_edge": rgb_edge,
        "depth_edge": depth_edge,
    }


def grabcut_refine(rgb, coarse_mask, roi_box, cam):
    """Optional RGB-aware boundary refinement initialized by the grid mask and ROI.

    This is a classical baseline only. If its initialization looks unreliable or OpenCV fails,
    the coarse grid mask is returned unchanged.
    """
    if not ENABLE_GRABCUT_REFINEMENT:
        return coarse_mask.astype(bool), "disabled"
    height, width = rgb.shape[:2]
    coarse = coarse_mask.astype(np.uint8)
    area = float(coarse.mean())
    if area < 0.002 or area > 0.90:
        return coarse.astype(bool), "skipped: implausible coarse-mask area"

    try:
        x0, y0, x1, y1 = roi_box
        roi = np.zeros((height, width), dtype=bool)
        roi[max(0, y0):min(height, y1), max(0, x0):min(width, x1)] = True
        if roi.mean() < 0.01:
            return coarse.astype(bool), "skipped: ROI too small"

        radius = max(3, int(round(min(height, width) * 0.015)))
        kernel_size = max(3, 2 * radius + 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        dilated = cv2.dilate(coarse, kernel, iterations=1).astype(bool) & roi
        eroded = cv2.erode(coarse, np.ones((3, 3), np.uint8), iterations=1).astype(bool)

        mask_labels = np.full((height, width), cv2.GC_PR_BGD, dtype=np.uint8)
        mask_labels[~roi] = cv2.GC_BGD
        mask_labels[roi & dilated] = cv2.GC_PR_FGD
        mask_labels[roi & ~dilated] = cv2.GC_PR_BGD
        mask_labels[eroded & roi] = cv2.GC_FGD

        if not np.any(mask_labels == cv2.GC_FGD):
            cam_roi = cam[roi]
            if cam_roi.size:
                threshold = float(np.percentile(cam_roi, 96))
                sure_fg = roi & (cam >= threshold)
                mask_labels[sure_fg] = cv2.GC_FGD
        if not np.any(mask_labels == cv2.GC_FGD):
            return coarse.astype(bool), "fallback: no reliable foreground seed"

        if not np.any(mask_labels == cv2.GC_BGD):
            border = np.zeros((height, width), dtype=bool)
            border[:2, :] = True
            border[-2:, :] = True
            border[:, :2] = True
            border[:, -2:] = True
            mask_labels[border] = cv2.GC_BGD

        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        bg_model = np.zeros((1, 65), dtype=np.float64)
        fg_model = np.zeros((1, 65), dtype=np.float64)
        cv2.grabCut(bgr, mask_labels, None, bg_model, fg_model, GRABCUT_ITERATIONS, cv2.GC_INIT_WITH_MASK)
        refined = np.isin(mask_labels, [cv2.GC_FGD, cv2.GC_PR_FGD])
        refined_area = float(refined.mean())
        if refined_area < 0.002 or refined_area > 0.90:
            return coarse.astype(bool), "fallback: refined area outside safety bounds"
        return refined, "GrabCut refinement applied"
    except Exception as exc:
        return coarse.astype(bool), f"fallback: GrabCut error ({type(exc).__name__})"


def save_binary_mask(mask, path):
    arr = (np.asarray(mask).astype(np.uint8) * 255)
    Image.fromarray(arr, mode="L").save(path)


def heatmap_rgb(values, palette=cv2.COLORMAP_TURBO):
    values_u8 = (np.clip(values, 0, 1) * 255).astype(np.uint8)
    return cv2.cvtColor(cv2.applyColorMap(values_u8, palette), cv2.COLOR_BGR2RGB)


def draw_detection_overlay(rgb, detections, selected=None):
    out = Image.fromarray(rgb.copy())
    draw = ImageDraw.Draw(out)
    try:
        font = ImageFont.truetype("arial.ttf", max(10, rgb.shape[1] // 32))
    except Exception:
        font = ImageFont.load_default()
    for det in detections:
        x0, y0, x1, y1 = det["box"]
        is_selected = selected is not None and det["box"] == selected["box"] and det["label"] == selected["label"]
        color = (255, 72, 72) if is_selected else (80, 180, 255)
        draw.rectangle((x0, y0, x1, y1), outline=color, width=3 if is_selected else 1)
        draw.text((x0 + 2, max(0, y0 - 14)), f"{det['label']} {det['score']:.2f}", fill=color, font=font)
    return np.asarray(out)


def grid_visualization(rgb, cam, depth_norm, cell_mask, roi_box):
    """Display-only fusion of depth and importance with a grid. Never fed to a model."""
    h, w = rgb.shape[:2]
    cam_display = heatmap_rgb(cam, cv2.COLORMAP_TURBO)
    depth_display = heatmap_rgb(depth_norm, cv2.COLORMAP_INFERNO)
    base = cv2.addWeighted(cam_display, 0.55, depth_display, 0.45, 0)
    overlay = base.copy()
    # Upscale selected grid cells to a pixel mask for display tinting only.
    cell_mask_img = np.zeros((h, w), dtype=np.uint8)
    ye = np.linspace(0, h, GRID_SIZE + 1, dtype=int)
    xe = np.linspace(0, w, GRID_SIZE + 1, dtype=int)
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if cell_mask[r, c]:
                cell_mask_img[ye[r]:ye[r+1], xe[c]:xe[c+1]] = 1
    tint = np.zeros_like(overlay)
    tint[:] = (255, 215, 0)
    overlay[cell_mask_img.astype(bool)] = (0.45 * overlay[cell_mask_img.astype(bool)] + 0.55 * tint[cell_mask_img.astype(bool)]).astype(np.uint8)
    for y in ye:
        cv2.line(overlay, (0, int(y)), (w - 1, int(y)), (245, 245, 245), 1)
    for x in xe:
        cv2.line(overlay, (int(x), 0), (int(x), h - 1), (245, 245, 245), 1)
    x0, y0, x1, y1 = roi_box
    cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 255, 255), 2)
    return overlay


def contour_overlay(rgb, mask, roi_box=None):
    out = rgb.copy()
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    thickness = max(1, int(round(min(rgb.shape[:2]) / 100)))
    cv2.drawContours(out, contours, -1, (40, 255, 120), thickness)
    if roi_box is not None:
        x0, y0, x1, y1 = roi_box
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 220, 255), 1)
    return out

print("Helper functions defined.")

## Cell 6: Run the isolation pipeline

In [ ]:
#Cell 6: Intent
# Run inference on the fixed sample, save each signal independently, and record per-stage timings.

per_image_rows = []
detection_rows = []
error_rows = []

for record in tqdm(records, desc="Bolero subject-isolation pilot"):
    item_id = record["item_id"]
    image_path = Path(record["path"])
    try:
        image = Image.open(image_path).convert("RGB")
        rgb = np.asarray(image, dtype=np.uint8)
        h, w = rgb.shape[:2]

        # 1) Classification + importance map.
        t0 = time.perf_counter()
        top5, cam = resnet_prediction_and_gradcam(image)
        resnet_seconds = time.perf_counter() - t0
        top1 = top5[0]
        class_index = int(top1["class_id"])
        predicted_label = str(top1["label"])

        # 2) Relative depth. The palette is display-only and is not model input.
        t0 = time.perf_counter()
        depth_norm, depth_raw = predict_depth(image)
        depth_seconds = time.perf_counter() - t0

        # 3) Object detections. DETR has the COCO label ontology, which does not cover every class.
        t0 = time.perf_counter()
        detections = predict_detections(image)
        detr_seconds = time.perf_counter() - t0
        selected_det, selection_method = select_subject_detection(detections, cam, predicted_label, w, h)

        # 4) Define a loose ROI. A selected DETR box is expanded slightly to allow contour refinement.
        if selected_det is not None:
            raw_selected_box = selected_det["box"]
            roi_box = expand_box(raw_selected_box, w, h, ROI_EXPANSION)
        else:
            raw_selected_box = None
            roi_box = fallback_cam_box(cam)

        # 5) Grid-based importance seeding and RGB/depth boundary-aware region growth.
        t0 = time.perf_counter()
        proposal = make_grid_proposal(rgb, cam, depth_norm, roi_box, grid_size=GRID_SIZE)
        grid_seconds = time.perf_counter() - t0
        grid_mask = proposal["mask"]

        # 6) Optional pixel-level classical refinement. It cannot invent object pixels semantically;
        # it only tests whether RGB appearance plus our coarse initialization yields a better contour.
        t0 = time.perf_counter()
        refined_mask, refinement_status = grabcut_refine(rgb, grid_mask, roi_box, cam)
        refine_seconds = time.perf_counter() - t0

        # Save the unmodified RGB source and each diagnostic product to this run's isolated output.
        base_path = OUTPUT_DIRS["original"] / f"{item_id}_original.png"
        image.save(base_path)
        Image.fromarray((depth_norm * 255).astype(np.uint8), mode="L").save(OUTPUT_DIRS["depth"] / f"{item_id}_relative_depth_gray.png")
        Image.fromarray(heatmap_rgb(depth_norm)).save(OUTPUT_DIRS["depth"] / f"{item_id}_pseudo_thermal_depth.png")
        luminance = normalize_robust(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), 1, 99)
        Image.fromarray((luminance * 255).astype(np.uint8), mode="L").save(OUTPUT_DIRS["illumination"] / f"{item_id}_rgb_luminance_gray.png")
        Image.fromarray(heatmap_rgb(luminance, cv2.COLORMAP_INFERNO)).save(OUTPUT_DIRS["illumination"] / f"{item_id}_rgb_luminance_pseudo_thermal.png")
        Image.fromarray((cam * 255).astype(np.uint8), mode="L").save(OUTPUT_DIRS["importance"] / f"{item_id}_gradcam_gray.png")
        Image.fromarray(heatmap_rgb(cam, cv2.COLORMAP_TURBO)).save(OUTPUT_DIRS["importance"] / f"{item_id}_gradcam_heatmap.png")
        cam_overlay = cv2.addWeighted(rgb, 0.58, heatmap_rgb(cam), 0.42, 0)
        Image.fromarray(cam_overlay).save(OUTPUT_DIRS["importance"] / f"{item_id}_gradcam_overlay.png")

        detection_vis = draw_detection_overlay(rgb, detections, selected_det)
        Image.fromarray(detection_vis).save(OUTPUT_DIRS["detections"] / f"{item_id}_detr_boxes.png")
        save_binary_mask(grid_mask, OUTPUT_DIRS["masks"] / f"{item_id}_grid_mask.png")
        save_binary_mask(refined_mask, OUTPUT_DIRS["masks"] / f"{item_id}_refined_mask.png")
        grid_vis = grid_visualization(rgb, cam, depth_norm, proposal["cell_mask"], roi_box)
        Image.fromarray(grid_vis).save(OUTPUT_DIRS["masks"] / f"{item_id}_importance_grid_selection.png")
        final_overlay = contour_overlay(rgb, refined_mask, roi_box)
        Image.fromarray(final_overlay).save(OUTPUT_DIRS["overlays"] / f"{item_id}_boundary_overlay.png")

        # Save one visual audit panel per image so all stages can be compared at a glance.
        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        axes = axes.flatten()
        panels = [
            (rgb, "Original RGB (model input kept untouched)"),
            (heatmap_rgb(depth_norm), "Relative depth (pseudo-thermal display)"),
            (cam_overlay, f"ResNet Grad-CAM: {predicted_label} ({top1['score']:.2f})"),
            (detection_vis, "DETR boxes (selected box highlighted)"),
            (grid_vis, f"{GRID_SIZE}×{GRID_SIZE} depth + importance grid"),
            (final_overlay, f"Contour proposal: {refinement_status}"),
        ]
        for ax, (panel_img, title) in zip(axes, panels):
            ax.imshow(panel_img)
            ax.set_title(title, fontsize=9)
            ax.axis("off")
        fig.suptitle(f"{item_id} | folder label for reporting only: {record['folder_label']}", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIRS["overlays"] / f"{item_id}_pipeline_summary.png", dpi=115, bbox_inches="tight")
        plt.close(fig)

        for det_i, det in enumerate(detections):
            detection_rows.append({
                "item_id": item_id,
                "detection_rank": det_i + 1,
                "label": det["label"],
                "label_id": det["label_id"],
                "score": det["score"],
                "x0": det["box"][0], "y0": det["box"][1],
                "x1": det["box"][2], "y1": det["box"][3],
                "cam_mass_fraction": box_cam_mass(cam, det["box"]),
                "selected": bool(selected_det is not None and det["box"] == selected_det["box"] and det["label"] == selected_det["label"]),
            })

        gt_label = IMAGENETTE_LABELS.get(record["folder_synset"], None)
        gt_index = imagenet_category_to_index.get(gt_label.lower()) if gt_label else None
        per_image_rows.append({
            "item_id": item_id,
            "image_path": str(image_path),
            "folder_synset": record["folder_synset"],
            "folder_label_reporting_only": record["folder_label"],
            "resnet_top1": predicted_label,
            "resnet_top1_class_id": class_index,
            "resnet_top1_confidence": top1["score"],
            "resnet_top5": " | ".join(f"{p['label']} ({p['score']:.3f})" for p in top5),
            "resnet_top1_correct_vs_folder_label": (class_index == gt_index) if gt_index is not None else None,
            "detr_detection_count": len(detections),
            "selected_detr_label": selected_det["label"] if selected_det else None,
            "selected_detr_score": selected_det["score"] if selected_det else None,
            "selected_detr_cam_mass_fraction": selected_det["cam_mass_fraction"] if selected_det else None,
            "selection_method": selection_method,
            "roi_x0": roi_box[0], "roi_y0": roi_box[1], "roi_x1": roi_box[2], "roi_y1": roi_box[3],
            "grid_cells_selected": int(proposal["cell_mask"].sum()),
            "grid_mask_area_fraction": float(grid_mask.mean()),
            "refined_mask_area_fraction": float(refined_mask.mean()),
            "pixel_refinement_status": refinement_status,
            "resnet_seconds": resnet_seconds,
            "depth_seconds": depth_seconds,
            "detr_seconds": detr_seconds,
            "grid_seconds": grid_seconds,
            "refinement_seconds": refine_seconds,
            "total_seconds": resnet_seconds + depth_seconds + detr_seconds + grid_seconds + refine_seconds,
        })

    except Exception as exc:
        error_rows.append({
            "item_id": item_id,
            "image_path": str(image_path),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(limit=5),
        })
        print(f"\nFAILED: {item_id} | {type(exc).__name__}: {exc}")

per_image_df = pd.DataFrame(per_image_rows)
detections_df = pd.DataFrame(detection_rows)
errors_df = pd.DataFrame(error_rows)
per_image_df.to_csv(OUTPUT_DIRS["reports"] / "per_image_results.csv", index=False)
detections_df.to_csv(OUTPUT_DIRS["reports"] / "all_detr_detections.csv", index=False)
errors_df.to_csv(OUTPUT_DIRS["reports"] / "errors.csv", index=False)

print(f"\nCompleted successfully: {len(per_image_df)} / {len(records)} images")
print(f"Errors: {len(errors_df)}")
print(f"Outputs saved to: {OUTPUT_ROOT}")
if len(per_image_df) == 0:
    raise RuntimeError("No images completed. Inspect reports/errors.csv and the first exception above.")

## Cell 7: Summarize the first isolation run

In [ ]:
#Cell 7: Intent
# Summarize model detections, proposal area, timings, and class-level baseline scores without overclaiming mask accuracy.

summary = {
    "images_attempted": len(records),
    "images_completed": len(per_image_df),
    "images_failed": len(errors_df),
    "grid_size": GRID_SIZE,
    "device": str(DEVICE),
    "depth_model": DEPTH_MODEL_ID,
    "detr_model": DETR_MODEL_ID,
    "resnet_weights": str(RESNET_WEIGHTS),
    "mean_detr_detections_per_image": float(per_image_df["detr_detection_count"].mean()) if len(per_image_df) else None,
    "selected_detr_box_rate": float(per_image_df["selected_detr_label"].notna().mean()) if len(per_image_df) else None,
    "mean_grid_mask_area_fraction": float(per_image_df["grid_mask_area_fraction"].mean()) if len(per_image_df) else None,
    "median_grid_mask_area_fraction": float(per_image_df["grid_mask_area_fraction"].median()) if len(per_image_df) else None,
    "mean_refined_mask_area_fraction": float(per_image_df["refined_mask_area_fraction"].mean()) if len(per_image_df) else None,
    "mean_total_seconds_per_image": float(per_image_df["total_seconds"].mean()) if len(per_image_df) else None,
}
if len(per_image_df) and per_image_df["resnet_top1_correct_vs_folder_label"].notna().any():
    known = per_image_df["resnet_top1_correct_vs_folder_label"].dropna().astype(bool)
    summary["resnet_full_frame_top1_accuracy_on_sample"] = float(known.mean()) if len(known) else None
else:
    summary["resnet_full_frame_top1_accuracy_on_sample"] = None

with open(OUTPUT_DIRS["reports"] / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(pd.Series(summary).to_string())

if "folder_label_reporting_only" in per_image_df.columns:
    class_summary = per_image_df.groupby("folder_label_reporting_only", dropna=False).agg(
        images=("item_id", "count"),
        mean_grid_mask_area=("grid_mask_area_fraction", "mean"),
        mean_refined_mask_area=("refined_mask_area_fraction", "mean"),
        selected_box_rate=("selected_detr_label", lambda s: s.notna().mean()),
        mean_total_seconds=("total_seconds", "mean"),
    ).reset_index()
    class_summary.to_csv(OUTPUT_DIRS["reports"] / "class_summary.csv", index=False)
    display(class_summary.round(3))

print("\nA mask area or detection rate is a diagnostic, not evidence of correct segmentation.")
print("For actual boundary quality, annotate a representative subset and run the optional evaluation cell below.")

## Cell 8: Inspect selected images

In [ ]:
#Cell 8: Intent
# Display a few saved per-image audit panels inside the notebook.

from IPython.display import display, Image as IPyImage

N_TO_DISPLAY = min(6, len(per_image_df))
for _, row in per_image_df.head(N_TO_DISPLAY).iterrows():
    summary_path = OUTPUT_DIRS["overlays"] / f"{row['item_id']}_pipeline_summary.png"
    print(f"{row['item_id']} | predicted: {row['resnet_top1']} | selected detection: {row['selected_detr_label']} | mask area: {row['refined_mask_area_fraction']:.1%}")
    display(IPyImage(filename=str(summary_path)))

## Cell 9: Optional manual boundary annotation and IoU evaluation

Automatic masks are proposals, not ground truth. This optional helper lets us create a small, manually traced reference mask for selected images. Click around the subject boundary in order, then press **Enter** to finish the polygon. Masks are stored under `annotations/ground_truth_masks/`, outside the output folder that gets cleared on the next run.

After annotating several images, run `evaluate_manual_masks()` to compare the coarse grid proposal and the refined mask using IoU and Dice. This is the first way to measure whether the contour is actually close to the visible subject boundary.

In [ ]:
#Cell 9: Intent
# Optional tools for manual ground-truth polygons and objective mask-overlap metrics.

from matplotlib.path import Path as MatplotlibPath


def annotate_mask(item_id):
    """Manually trace one visible subject polygon and save a binary reference mask.

    Usage: annotate_mask("n02102040_example_name") using an item_id from per_image_results.csv.
    Click boundary points clockwise/counter-clockwise, then press Enter. This draws one polygon;
    use separate image annotations later if a subject contains disconnected visible components.
    """
    original_path = OUTPUT_DIRS["original"] / f"{item_id}_original.png"
    if not original_path.exists():
        raise FileNotFoundError(f"Cannot find saved original for item_id={item_id}. Check outputs/reports/per_image_results.csv")
    rgb = np.asarray(Image.open(original_path).convert("RGB"))
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(rgb)
    ax.set_title(f"Trace subject boundary: {item_id}\nClick points around the outline, then press Enter")
    ax.axis("on")
    points = plt.ginput(n=-1, timeout=0, show_clicks=True)
    plt.close(fig)
    if len(points) < 3:
        print("At least three points are needed. No mask saved.")
        return None
    yy, xx = np.mgrid[0:rgb.shape[0], 0:rgb.shape[1]]
    coords = np.vstack([xx.ravel(), yy.ravel()]).T
    polygon = MatplotlibPath(np.asarray(points, dtype=np.float32))
    mask = polygon.contains_points(coords).reshape(rgb.shape[0], rgb.shape[1])
    destination = ANNOTATION_ROOT / f"{item_id}_mask.png"
    save_binary_mask(mask, destination)
    print(f"Saved manual reference mask: {destination}")
    return destination


def _mask_scores(predicted, reference):
    p = np.asarray(predicted).astype(bool)
    g = np.asarray(reference).astype(bool)
    if p.shape != g.shape:
        p = cv2.resize(p.astype(np.uint8), (g.shape[1], g.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
    intersection = int(np.logical_and(p, g).sum())
    union = int(np.logical_or(p, g).sum())
    p_sum, g_sum = int(p.sum()), int(g.sum())
    iou = intersection / union if union else 1.0
    dice = (2 * intersection) / (p_sum + g_sum) if (p_sum + g_sum) else 1.0
    return iou, dice


def evaluate_manual_masks():
    """Compare saved masks with manual masks and write per-image IoU/Dice results."""
    rows = []
    for gt_path in sorted(ANNOTATION_ROOT.glob("*_mask.png")):
        item_id = gt_path.name[:-len("_mask.png")]
        gt = np.asarray(Image.open(gt_path).convert("L")) > 0
        candidates = {
            "grid": OUTPUT_DIRS["masks"] / f"{item_id}_grid_mask.png",
            "refined": OUTPUT_DIRS["masks"] / f"{item_id}_refined_mask.png",
        }
        for method, pred_path in candidates.items():
            if not pred_path.exists():
                continue
            pred = np.asarray(Image.open(pred_path).convert("L")) > 0
            iou, dice = _mask_scores(pred, gt)
            rows.append({"item_id": item_id, "method": method, "IoU": iou, "Dice": dice, "reference_pixels": int(gt.sum()), "predicted_pixels": int(pred.sum())})
    results = pd.DataFrame(rows)
    if results.empty:
        print(f"No matching annotations found. Use annotate_mask(item_id) first. Annotation folder: {ANNOTATION_ROOT}")
        return results
    results.to_csv(OUTPUT_DIRS["reports"] / "manual_mask_metrics.csv", index=False)
    display(results.round(3))
    display(results.groupby("method")[["IoU", "Dice"]].agg(["mean", "median", "count"]).round(3))
    return results

print("Optional tools ready. Example: annotate_mask(per_image_df.iloc[0]['item_id']); then evaluate_manual_masks().")

## How to interpret this run

- `*_pseudo_thermal_depth.png` is a palette applied to **relative depth** for easier inspection; it is not a thermal camera output.
- `*_gradcam_heatmap.png` and `*_gradcam_overlay.png` visualize regions that influence the current ResNet top-1 class. A CAM can be diffuse, fragmented, or wrong.
- `*_detr_boxes.png` shows raw DETR candidates. DETR is pretrained on COCO, so some Imagenette subjects are outside its label set. In that case the pipeline may use an importance-only ROI instead of pretending a box is valid.
- `*_importance_grid_selection.png` shows a coarse, connected region grown from high-importance seed cells while discouraging growth across strong RGB/depth discontinuities. This is a hypothesis about the subject region, not a segmentation guarantee.
- `*_refined_mask.png` is the optional GrabCut result initialized from the grid mask. It may improve or worsen the outline; compare it with the grid mask and manual annotations.
- `*_boundary_overlay.png` draws the resulting contour on a copy of the original for inspection only. The original file used for model inference stays untouched.
- `per_image_results.csv` keeps per-stage timings separate. Do not interpret a high mask area, confidence score, or attractive overlay as proof of correct isolation.

## First milestone / exit criteria

Before we add image reconstruction, inspect a representative subset and collect manual masks for approximately 30–50 images spanning classes and failure cases. Compare grid-only, refined, and simpler baselines using IoU and Dice. Also report failure examples and class coverage. Fix the region proposal and data protocol before introducing a local generative model.